# MIHIC FP32 and INT8 comparison across three seeds

Open this notebook from the repository root after training. Install the packages listed in the [README](README.md#5-reproduce-the-int8-experiment) in the notebook's Python environment.

Set `DATA_ROOT` to the directory containing `train/`, `val/`, and `test/`. Set `RUN_ROOT` to the training output directory containing `runs/`. The cell uses the three Small single + KD checkpoints for seeds 42, 123, and 2026 and evaluates FP32 and INT8 on the official test split.

Run the cell to generate Table 6 and Figure 6. Outputs go to a timestamped subdirectory of `outputs/MIHIC_INT8/`. The final line gives the path to the results ZIP.


In [ ]:
import sys
from pathlib import Path

print("Kernel ready:", sys.executable, flush=True)

PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT / "dataset"
RUN_ROOT = PROJECT_ROOT / "MIHIC_reproduction_output"

# Point DATA_ROOT and RUN_ROOT to your local dataset and completed training run.
DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()
RUN_ROOT = Path(RUN_ROOT).expanduser().resolve()
TRAINING_SCRIPT = PROJECT_ROOT / "mihic_pipeline.py"
MODEL_NAME = "small_single_KD"
SEEDS = (42, 123, 2026)
CHECKPOINTS = {seed: RUN_ROOT / "runs" / MODEL_NAME / f"seed_{seed}" / "best.pt" for seed in SEEDS}

required_dirs = [DATA_ROOT / split for split in ("train", "val", "test")]
required_files = [TRAINING_SCRIPT, *CHECKPOINTS.values()]
for path in [*required_dirs, *required_files]:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}", flush=True)

if not all(p.is_dir() for p in required_dirs) or not all(p.is_file() for p in required_files):
    raise FileNotFoundError("Open this notebook from the repository root and check DATA_ROOT and RUN_ROOT.")

from importlib.util import find_spec

missing = [name for name in ("onnx", "onnxruntime") if find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        "Install ONNX dependencies in this kernel's environment: "
        "python -m pip install onnx==1.23.2 onnxruntime==1.31.1"
    )

print("Importing PyTorch...", flush=True)
import torch
import torchvision
print("Importing ONNX and ONNX Runtime...", flush=True)
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from onnxruntime.quantization import (
    CalibrationDataReader, CalibrationMethod, QuantFormat, QuantType, quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from sklearn.metrics import precision_recall_fscore_support

PROVIDER = "CPUExecutionProvider"
if PROVIDER not in ort.get_available_providers():
    raise RuntimeError(f"{PROVIDER} is unavailable.")
THREADS = 8
torch.set_num_threads(THREADS)
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("ONNX:", onnx.__version__, "| ONNX Runtime:", ort.__version__)
print("Providers:", ort.get_available_providers(), "| CPU threads:", THREADS)
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, StrMethodFormatter
from IPython.display import display, Image


from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.util import module_from_spec, spec_from_file_location
import random

print("Loading the original model and preprocessing...", flush=True)
spec = spec_from_file_location("mihic_repository_pipeline", TRAINING_SCRIPT)
mihic = module_from_spec(spec)
sys.modules[spec.name] = mihic
spec.loader.exec_module(mihic)

N_CLASSES = len(mihic.CLASSES)

def check_partition(dataset, split):
    counts = Counter(dataset.targets)
    expected = mihic.EXPECTED[split]
    if dataset.classes != mihic.CLASSES or [counts[i] for i in range(N_CLASSES)] != expected:
        raise ValueError(f"{split} does not match the original MIHIC class mapping and counts.")

train, _, _ = mihic.create_datasets(DATA_ROOT, torch.device("cpu"), workers=0)
check_partition(train, "train")
groups = defaultdict(list)
for index, (path, label) in enumerate(train.samples):
    groups[mihic.analyze_stain(path), label].append(index)
if {stain for stain, _ in groups} != set(mihic.STAINS):
    raise ValueError("Unexpected stain labels in the training split.")

CALIBRATION_COUNT = 1000
CALIBRATION_SEED = 42
rng = random.Random(CALIBRATION_SEED)
selected = [rng.choice(groups[key]) for key in sorted(groups)]
used = set(selected)
selected.extend(rng.sample(
    [i for i in range(len(train)) if i not in used], CALIBRATION_COUNT - len(selected)
))
rng.shuffle(selected)
calibration_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(train, selected), batch_size=32, shuffle=False, num_workers=0,
)
example = next(iter(calibration_loader))[0][:8].contiguous()


stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULT_ROOT = PROJECT_ROOT / "outputs" / "MIHIC_INT8" / f"three_seeds_{stamp}"
RESULT_ROOT.mkdir(parents=True, exist_ok=False)
calibration_manifest = pd.DataFrame([
    {
        "sample_id": Path(train.samples[i][0]).relative_to(DATA_ROOT).as_posix(),
        "stain": mihic.analyze_stain(train.samples[i][0]),
        "tissue_class": mihic.CLASSES[train.samples[i][1]],
    }
    for i in selected
])
calibration_manifest.to_csv(RESULT_ROOT / "calibration_patches.csv", index=False)
print(f"Output directory: {RESULT_ROOT}", flush=True)
print(f"Calibration: {len(selected)} training patches across {len(groups)} stain/class combinations", flush=True)

from tempfile import TemporaryDirectory

OPSET = 17
INPUT, OUTPUT = "images", "logits"
def cpu_session(path, profile=False):
    options = ort.SessionOptions()
    options.intra_op_num_threads = THREADS
    options.inter_op_num_threads = 1
    options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    options.enable_profiling = profile
    if profile:
        options.profile_file_prefix = str(path.parent / "ort_int8_profile")
    session = ort.InferenceSession(str(path), sess_options=options, providers=[PROVIDER])
    if session.get_providers() != [PROVIDER]:
        raise RuntimeError(f"Unexpected execution providers: {session.get_providers()}")
    return session

def infer_logits(session, images):
    scores = session.run([OUTPUT], {INPUT: images})[0]
    if scores.shape != (len(images), N_CLASSES) or not np.isfinite(scores).all():
        raise ValueError(f"Invalid logits with shape {scores.shape}")
    return scores


import json

class TrainingCalibrationReader(CalibrationDataReader):
    def __init__(self, loader):
        self.loader = loader
        self.rewind()

    def rewind(self):
        self.batches = iter(self.loader)
        self.seen = 0

    def get_next(self):
        batch = next(self.batches, None)
        if batch is None:
            return None
        images, _, _ = batch
        self.seen += len(images)
        if self.seen % 256 == 0 or self.seen == CALIBRATION_COUNT:
            print(f"Calibration input: {self.seen}/{CALIBRATION_COUNT}", flush=True)
        return {INPUT: images.numpy()}


test = mihic.IDFolder(DATA_ROOT / "test", transform=train.transform)
check_partition(test, "test")
test_loader = torch.utils.data.DataLoader(test, batch_size=32, shuffle=False, num_workers=0)

from time import perf_counter


def run_seed(SEED, CHECKPOINT):
    print(f"\n=== Seed {SEED} ===", flush=True)
    OUT_DIR = RESULT_ROOT / f"seed_{SEED}"
    OUT_DIR.mkdir()
    checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    if checkpoint["model"] != MODEL_NAME or checkpoint["seed"] != SEED:
        raise ValueError("Checkpoint model or seed does not match the experiment.")
    if list(checkpoint["classes"]) != list(mihic.CLASSES):
        raise ValueError("Checkpoint and training script disagree on class order.")
    model, _ = mihic.make_model(MODEL_NAME, pretrained=False)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()

    FP32_MODEL = OUT_DIR / "model_fp32.onnx"
    print("Exporting PyTorch to FP32 ONNX...", flush=True)
    with TemporaryDirectory(dir=OUT_DIR) as work:
        raw = Path(work) / "export.onnx"
        with torch.inference_mode():
            torch.onnx.export(
                model, (example[:1],), str(raw),
                input_names=[INPUT], output_names=[OUTPUT],
                dynamic_axes={INPUT: {0: "batch"}, OUTPUT: {0: "batch"}},
                opset_version=OPSET, dynamo=False, external_data=False,
            )
            torch_logits = model(example).numpy()
        print("Preparing the graph for quantization...", flush=True)
        quant_pre_process(input_model=raw, output_model_path=FP32_MODEL)
    onnx.checker.check_model(str(FP32_MODEL))
    fp32_session = cpu_session(FP32_MODEL)
    fp32_logits = infer_logits(fp32_session, example.numpy())
    max_logit_diff = float(np.max(np.abs(torch_logits - fp32_logits)))
    print(f"Maximum absolute logit difference: {max_logit_diff:.6g}")
    np.testing.assert_allclose(fp32_logits, torch_logits, atol=1e-4, rtol=1e-4)
    print("FP32 export check passed on 8 training patches.")

    INT8_MODEL = OUT_DIR / "model_int8.onnx"
    print("Running static INT8 quantization...", flush=True)
    quantize_static(
        str(FP32_MODEL), str(INT8_MODEL), TrainingCalibrationReader(calibration_loader),
        quant_format=QuantFormat.QDQ, activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8, per_channel=True, calibrate_method=CalibrationMethod.MinMax,
    )
    int8_graph = onnx.load(str(INT8_MODEL))
    onnx.checker.check_model(int8_graph)
    if not any(t.data_type == onnx.TensorProto.INT8 for t in int8_graph.graph.initializer):
        raise RuntimeError("INT8 weights were not found in the quantized graph.")
    del int8_graph

    profiling_session = cpu_session(INT8_MODEL, profile=True)
    infer_logits(profiling_session, example.numpy())
    profile_path = Path(profiling_session.end_profiling())
    del profiling_session
    events = json.loads(profile_path.read_text())
    ops = Counter(
        (event["args"]["provider"], event["args"]["op_name"])
        for event in events
        if event.get("cat") == "Node"
        and event.get("name", "").endswith("_kernel_time")
        and {"provider", "op_name"} <= event.get("args", {}).keys()
    )
    if not ops:
        raise RuntimeError("The ONNX Runtime profile contains no operator events.")
    operator_table = pd.DataFrame([
        {"provider": provider, "operator": op, "calls": count}
        for (provider, op), count in sorted(ops.items())
    ])
    operator_table.to_csv(OUT_DIR / "int8_executed_operators.csv", index=False)
    integer_ops = sorted({op for _, op in ops if op.startswith("QLinear") or "Integer" in op or op == "QGemm"})
    if not integer_ops:
        raise RuntimeError("No integer kernels identified. Inspect int8_executed_operators.csv before proceeding.")
    int8_session = cpu_session(INT8_MODEL)
    print("Integer operators observed:", ", ".join(integer_ops))
    print("INT8 smoke test passed.")

    sessions = {"FP32": fp32_session, "INT8": int8_session}
    outputs = {name: [] for name in sessions}
    targets = []
    seen = 0

    print(f"Evaluating both models on {len(test):,} test patches...", flush=True)
    for batch, (images, labels, _) in enumerate(test_loader, start=1):
        inputs = images.numpy()
        targets.append(labels.numpy())
        for name, session in sessions.items():
            outputs[name].append(infer_logits(session, inputs))
        seen += len(labels)
        if batch % 50 == 0 or seen == len(test):
            print(f"Test patches: {seen:,}/{len(test):,}", flush=True)

    y_test = np.concatenate(targets)
    metric_rows, class_rows = [], []
    for name, pieces in outputs.items():
        probabilities = mihic.softmax_np(np.concatenate(pieces))
        all_metrics = mihic.accuracy_calibration(y_test, probabilities)
        metric_rows.append({
            "precision": name,
            **{metric: all_metrics[metric] for metric in ("macro_f1", "accuracy", "nll", "ece15")},
        })
        _, _, class_f1, support = precision_recall_fscore_support(
            y_test, probabilities.argmax(axis=1), labels=range(N_CLASSES), zero_division=0,
        )
        class_rows.extend(
            {"precision": name, "tissue_class": label, "f1": score, "test_patches": int(count)}
            for label, score, count in zip(mihic.CLASSES, class_f1, support)
        )
    metrics = pd.DataFrame(metric_rows).set_index("precision")
    metrics.to_csv(OUT_DIR / "ptq_test_metrics.csv")
    pd.DataFrame(class_rows).to_csv(OUT_DIR / "ptq_per_class.csv", index=False)
    print(metrics.round(6).to_string(), flush=True)

    benchmark_settings = {1: (10, 40), 64: (5, 20)}
    benchmark_rng = np.random.default_rng(42)
    benchmark_rows, timing_rows = [], []

    for batch_size, (warmup, repeats) in benchmark_settings.items():
        inputs = benchmark_rng.standard_normal((batch_size, 3, 128, 128), dtype=np.float32)
        feed = {INPUT: inputs}
        names = tuple(sessions)
        for session in sessions.values():
            for _ in range(warmup):
                session.run([OUTPUT], feed)
        print(f"Benchmarking batch size {batch_size}...", flush=True)
        for repeat in range(repeats):
            for name in (names if repeat % 2 == 0 else names[::-1]):
                start = perf_counter()
                sessions[name].run([OUTPUT], feed)
                timing_rows.append({
                    "precision": name, "batch_size": batch_size, "repeat": repeat + 1,
                    "latency_ms": 1000 * (perf_counter() - start),
                })

    timings = pd.DataFrame(timing_rows)
    timings.to_csv(OUT_DIR / "ptq_cpu_timings.csv", index=False)
    median = timings.groupby(["precision", "batch_size"])["latency_ms"].median()
    for name, path in (("FP32", FP32_MODEL), ("INT8", INT8_MODEL)):
        benchmark_rows.append({
            "precision": name, "onnx_bytes": path.stat().st_size,
            "cpu_batch1_latency_ms": median.loc[name, 1],
            "cpu_batch64_images_per_s": 64000 / median.loc[name, 64],
        })
    benchmark = pd.DataFrame(benchmark_rows).set_index("precision")
    benchmark.to_csv(OUT_DIR / "ptq_cpu_benchmark.csv")
    print(benchmark.round(6).to_string(), flush=True)

    import platform

    summary = metrics.join(benchmark)
    summary.insert(summary.columns.get_loc("onnx_bytes") + 1, "onnx_MB", summary["onnx_bytes"] / 1e6)
    summary.loc["INT8 minus FP32"] = summary.loc["INT8"] - summary.loc["FP32"]
    summary.to_csv(OUT_DIR / "ptq_summary.csv")

    cpu_info = Path("/proc/cpuinfo")
    cpu_lines = cpu_info.read_text().splitlines() if cpu_info.is_file() else []
    cpu_name = next((line.split(":", 1)[1].strip() for line in cpu_lines
                     if line.startswith("model name")), platform.processor())
    protocol = {
        "model": MODEL_NAME, "seed": SEED, "checkpoint_epoch": checkpoint["epoch"],
        "checkpoint_sha256": mihic.sha256file(CHECKPOINT),
        "training_script_sha256": mihic.sha256file(TRAINING_SCRIPT),
        "checkpoint_config_sha256": checkpoint.get("config_sha256"),
        "calibration_split": "train", "calibration_count": len(calibration_manifest),
        "calibration_seed": CALIBRATION_SEED, "test_count": len(y_test),
        "test_probability_temperature": 1.0,
        "quantization": "static QDQ, signed INT8 weights and activations, per-channel weights, MinMax",
        "opset": OPSET, "export_check_patches": len(example),
        "fp32_export_max_abs_logit_diff": max_logit_diff, "export_atol": 1e-4, "export_rtol": 1e-4,
        "integer_operators_observed": integer_ops,
        "provider": PROVIDER, "intra_op_threads": THREADS, "inter_op_threads": 1,
        "cpu": cpu_name, "platform": platform.platform(), "python": platform.python_version(),
        "torch": str(torch.__version__), "torchvision": str(torchvision.__version__),
        "onnx": onnx.__version__, "onnxruntime": ort.__version__,
        "benchmark_input": "shared synthetic FP32 tensors, model-only, median latency",
        "benchmark_warmup_and_measured_calls": benchmark_settings,
        "benchmark_input_seed": 42,
    }
    (OUT_DIR / "ptq_protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")

for seed in SEEDS:
    run_seed(seed, CHECKPOINTS[seed])
    print(f"Completed seed {seed} in {RESULT_ROOT / f'seed_{seed}'}", flush=True)


# Summarize the three paired checkpoint comparisons.
from zipfile import ZIP_DEFLATED, ZipFile

per_seed = pd.concat([
    pd.read_csv(RESULT_ROOT / f"seed_{seed}" / "ptq_summary.csv")
    .query("precision in ['FP32', 'INT8']").assign(seed=seed)
    for seed in SEEDS
], ignore_index=True)
per_seed = per_seed[["seed", "precision", *[c for c in per_seed if c not in ("seed", "precision")]]]
per_seed.to_csv(RESULT_ROOT / "ptq_by_seed.csv", index=False)
scores = per_seed.set_index(["precision", "seed"])
paired = scores.loc["INT8"] - scores.loc["FP32"]
paired.to_csv(RESULT_ROOT / "ptq_paired_int8_minus_fp32.csv")

metrics_for_paper = {
    "macro_f1": ("Test Macro-F1", 4),
    "accuracy": ("Test accuracy", 4),
    "nll": ("Test NLL", 4),
    "ece15": ("Test ECE (15 bins)", 4),
    "onnx_MB": ("ONNX size (MB)", 2),
    "cpu_batch1_latency_ms": ("CPU batch-1 latency (ms)", 3),
    "cpu_batch64_images_per_s": ("CPU batch-64 throughput (images/s)", 1),
}
table_rows, numeric_rows = [], []
for metric, (label, decimals) in metrics_for_paper.items():
    display_row, numeric = {"Metric": label}, {"metric": metric}
    for name, series in (
        ("FP32", scores.loc["FP32", metric]),
        ("INT8", scores.loc["INT8", metric]),
        ("INT8 minus FP32", paired[metric]),
    ):
        mean, sd = series.mean(), series.std(ddof=1)
        display_row[name] = f"{mean:.{decimals}f} ± {sd:.{decimals}f}"
        numeric[f"{name}_mean"], numeric[f"{name}_sd"] = mean, sd
    table_rows.append(display_row)
    numeric_rows.append(numeric)
table = pd.DataFrame(table_rows)
table.to_csv(RESULT_ROOT / "PTQ_Table_FP32_INT8.csv", index=False)
pd.DataFrame(numeric_rows).to_csv(RESULT_ROOT / "ptq_summary_mean_sd.csv", index=False)
display(table)

per_class = pd.concat([
    pd.read_csv(RESULT_ROOT / f"seed_{seed}" / "ptq_per_class.csv").assign(seed=seed)
    for seed in SEEDS
], ignore_index=True)
per_class.to_csv(RESULT_ROOT / "ptq_per_class_by_seed.csv", index=False)
class_scores = per_class.pivot(index=["seed", "tissue_class"], columns="precision", values="f1")
class_scores["INT8 minus FP32"] = class_scores["INT8"] - class_scores["FP32"]
class_scores.groupby(level="tissue_class").agg(["mean", "std"]).to_csv(
    RESULT_ROOT / "ptq_per_class_mean_sd.csv"
)

fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.5), layout="constrained")
plot_metrics = (
    ("macro_f1", "Test Macro-F1", "{x:.3f}"),
    ("onnx_MB", "ONNX file size (MB)", "{x:.1f}"),
    ("cpu_batch1_latency_ms", "CPU batch-1 latency (ms)", "{x:.2f}"),
    ("cpu_batch64_images_per_s", "CPU batch-64 throughput (images/s)", "{x:,.0f}"),
)
styles = {42: ("#0072B2", "o", -0.06), 123: ("#D55E00", "s", 0), 2026: ("#009E73", "^", 0.06)}
for letter, ax, (metric, title, fmt) in zip("ABCD", axes.flat, plot_metrics):
    for seed, (color, marker, offset) in styles.items():
        y = [scores.loc[(precision, seed), metric] for precision in ("FP32", "INT8")]
        ax.plot([offset, 1 + offset], y, marker=marker, color=color, linewidth=1.3,
                markersize=5.5, label=f"Seed {seed}")
    ax.set_title(f"{letter}  {title}", loc="left", fontsize=10, fontweight="bold")
    ax.set_xticks([0, 1], ["FP32", "INT8"])
    ax.set_xlim(-0.25, 1.25)
    if metric != "macro_f1":
        ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_formatter(StrMethodFormatter(fmt))
    ax.grid(axis="y", color="#E4E7EB", linewidth=0.7)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncols=3, frameon=False)
figure = RESULT_ROOT / "PTQ_Figure_FP32_INT8.png"
fig.savefig(figure, dpi=600, facecolor="white")
fig.savefig(RESULT_ROOT / "PTQ_Figure_FP32_INT8.pdf", facecolor="white")
plt.close(fig)
display(Image(filename=str(figure), width=740))

bundle = RESULT_ROOT / "MIHIC_INT8_ThreeSeeds_Results.zip"
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(RESULT_ROOT.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".csv", ".json", ".png", ".pdf"}:
            archive.write(path, path.relative_to(RESULT_ROOT).as_posix())
print("Results ZIP:", bundle, flush=True)